In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

: 

In [ ]:
df = pd.read_csv('cleaned_deceptive-opinion.csv')

In [ ]:
df.isnull().sum()

In [ ]:
tf.keras.utils.set_random_seed(42)

In [ ]:
df['target'] = df['deceptive'].map({'deceptive': 1, 'truthful': 0})
X = df['text']
y = df['target']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
vocab_size = 5000
max_length = 200
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train)

In [ ]:
x_train_padded = pad_sequences(tokenizer.texts_to_sequences(x_train), maxlen=max_length, padding='post')
x_test_padded = pad_sequences(tokenizer.texts_to_sequences(x_test), maxlen=max_length, padding='post')

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 16, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
history = model.fit(x_train_padded, y_train, epochs=10, validation_data=(x_test_padded, y_test))

In [ ]:
classification_report = model.predict(x_test_padded)
y_pred = (classification_report > 0.5).astype(int)
accuracy_score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy_score}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 5))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.xticks([0, 1], ['Truthful', 'Deceptive'])
plt.yticks([0, 1], ['Truthful', 'Deceptive'])
for i in range(len(cm)):
    for j in range(len(cm[i])):
        plt.text(j, i, cm[i][j], ha='center', va='center', color='red')
plt.show()

In [ ]:
model.save('nn_model.keras')
with open('nn_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)